# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [2]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [3]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [5]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [6]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [7]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [9]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [10]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [11]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Based on the information provided, exercises that can help with lower back pain include:\n\n1. Cat-Cow Stretch: Start on your hands and knees, then alternate between arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n\n2. Bird Dog: From hands and knees, extend your opposite arm and leg while keeping your core engaged. Hold each position for 5 seconds, then switch sides. Do 10 repetitions on each side.\n\n3. Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis slightly upwards. Hold for 10 seconds, then repeat 8-12 times.\n\nThese exercises are gentle stretches and strengthening movements designed to alleviate lower back discomfort and can help prevent future episodes.'

In [12]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in maintaining overall health. It is essential for physical recovery, as during sleep, your body repairs tissues and regenerates itself. Sleep also helps in consolidating memories and supports cognitive functions. Additionally, adequate sleep is important for regulating hormones that affect growth and appetite, boosting the immune system, reducing stress, and improving mental well-being. Consistently getting 7-9 hours of quality sleep can help prevent health issues such as weakened immunity, mental health problems, and cognitive impairments.'

In [13]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- For headaches:\n  - Drinking water to stay hydrated\n  - Applying cold or warm compresses to the head or neck\n  - Resting in a dark, quiet room\n  - Gentle massage of the temples and neck\n  - Using essential oils like peppermint or lavender\n  - Maintaining a regular sleep schedule\n  - Consuming small amounts of caffeine (with caution)\n\n- For stress relief:\n  - Deep breathing exercises (e.g., inhaling for 4 counts, holding, exhaling)\n  - Progressive muscle relaxation\n  - Grounding techniques (naming things you see, hear, feel, etc.)\n  - Taking short walks, preferably in nature\n  - Listening to calming music\n\nThese approaches can help manage stress and headaches naturally.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [14]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [15]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [16]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help alleviate lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, then alternate between arching your back up (cat pose) and letting it sag down (cow pose). Repeat 10-15 times.\n\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for about 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, tighten your abs, and tilt your pelvis slightly upward to flatten your lower back against the floor. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help manage and prevent lower back pain.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

"Sleep has a significant impact on overall health. Maintaining a consistent sleep schedule and creating a comfortable sleep environment—such as keeping the room cool, dark, and quiet—are important for promoting quality sleep. Good sleep hygiene practices include relaxing bedtime routines and limiting screen time, caffeine, and heavy meals before bed. Adequate sleep supports the body's immune function, mental health, and physical recovery, making it a foundational component of overall wellness."

In [18]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing relaxation techniques such as deep breathing and progressive muscle relaxation, engaging in meditation, and consuming herbal teas like chamomile or valerian root. Additionally, managing stress through activities like exercise and ensuring adequate hydration and sleep can help reduce headache triggers related to stress.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:
Example Query: "ModuleNotFoundError: No module named 'requests'"

1. Exact error text
The query is a literal error message. BM25 matches these tokens directly. Documents that contain this exact error and its fix (e.g., pip install requests) will rank highly. Embedding search often pulls generic "Python import errors" or "pip troubleshooting" content that may not address this specific error.

2. Token overlap is what matters
Users expect answers that mention this exact error and its cause. The important signal is the presence of "ModuleNotFoundError", "requests", and similar strings. BM25 rewards documents with those exact terms; embeddings may spread relevance across related but different error types.

3. Very domain-specific phrasing
Error messages like ModuleNotFoundError: No module named 'X' are highly formulaic. Embedding models may not treat them distinctly from other import or ModuleNotFoundError variants. BM25 treats each string as a literal token and does not conflate them with other error messages.

4. Common support/Stack Overflow–style queries
People usually copy-paste full error messages into search. BM25’s term matching is well suited to this: it directly rewards documents with the same wording. Semantic search can rank explanatory articles that don’t contain the exact error string, which is less useful for quick debugging.


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [19]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [20]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n1. **Cat-Cow Stretch:** Start on your hands and knees. Alternate arching your back up (cat pose) and letting it sag down (cow pose). Perform 10-15 repetitions.\n\n2. **Bird Dog:** From hands and knees, extend your opposite arm and leg simultaneously while engaging your core. Hold each extension for about 5 seconds, then switch sides. Do 10 repetitions per side.\n\n3. **Pelvic Tilts:** Lie on your back with knees bent. Flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and may prevent future episodes.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is essential for physical repair, mental well-being, and cognitive function. During sleep, your body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Getting enough quality sleep—generally 7-9 hours per night—supports these processes and helps maintain your overall health and wellness.'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing, progressive muscle relaxation, grounding techniques, taking short walks in nature, listening to calming music, staying hydrated by drinking water, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, gentle massage of the temples and neck, using essential oils like peppermint or lavender, and maintaining a regular sleep schedule.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [24]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [25]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [26]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off floor. Hold briefly, then lower. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese exercises can help alleviate discomfort and strengthen the muscles supporting your lower back. Ho

In [27]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is crucial for physical health, mental well-being, and cognitive function. During sleep, your body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep, typically 7-9 hours per night, supports immune function, helps maintain emotional stability, and enhances mental clarity. Conversely, poor sleep or sleep disturbances like insomnia can lead to health issues such as increased stress, impaired immune response, and greater risk of chronic conditions. Creating a consistent sleep routine and maintaining a conducive sleep environment are important practices for promoting overall health.'

In [28]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water and staying hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gently massaging the temples and neck\n- Using essential oils such as peppermint or lavender\n- Engaging in deep breathing, progressive muscle relaxation, or grounding techniques\n- Taking short walks, preferably in nature\n- Listening to calming music\n\nAdditionally, maintaining a regular sleep schedule, practicing mindfulness or meditation, managing stress through exercise and social connections, and engaging in hobbies can help reduce stress overall.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

Recall is the fraction of all relevant documents that your retrieval step actually returns. Higher recall means fewer relevant documents are missed.

The vocabulary mismatch problem
    People often phrase the same intent in different ways. Documents use varied terminology (synonyms, jargon, paraphrases). A single query will usually only match documents whose wording is close to that query, and will miss other relevant content that uses different phrasing.

How reformulations help
Reformulating the user query several times (e.g. with an LLM) produces alternative phrasings that:
    1. Target different language – e.g. “How do I reset my password?” vs “password recovery steps” vs “I forgot my login credentials”.
    2. Reflect different question styles – factual vs procedural vs conversational.
    3. Include synonyms and related concepts – broadening the set of terms that can match documents.

Why recall goes up
You typically run retrieval once per reformulation and then merge or deduplicate the results. Because each reformulation explores a different part of the semantic space, they often return different subsets of the relevant documents.
Query 1 might retrieve docs A, B, C
Query 2 might retrieve docs B, D, E
Query 3 might retrieve docs C, E, F
Union: A, B, C, D, E, F → more relevant documents overall than any single query would retrieve.


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [29]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [30]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [31]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [32]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [33]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [34]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'To help with lower back pain, the following exercises are recommended:\n\n- **Cat-Cow Stretch:** On hands and knees, alternate arching your back upward (cat) and sagging it down (cow). Perform 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend the opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over your chest, tighten your stomach muscles, and lift your shoulders off the floor. Repeat 8-12 times.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, tighten your abs, and tilt your pelvis upward slightly to flatten your back against the floor. Hold for 10 seconds, and repeat 8-12 times.\n\nThese gentle exercises can help alleviate lower back discomfort and may prevent future 

In [35]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep positively affects overall health by supporting physical repair, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7-9 hours per night—helps maintain energy levels, immune function, and emotional stability. Poor sleep or insufficient sleep can lead to fatigue, impaired concentration, weakened immune response, and increased risk of chronic conditions. Therefore, prioritizing quality sleep through good sleep hygiene practices is essential for overall health and wellness.'

In [36]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

"Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises to help relax the body and mind.\n- Progressive muscle relaxation to release tension.\n- Grounding techniques, such as noticing five things you see, four you hear, three you feel, two you smell, and one you taste.\n- Taking short walks, especially in nature, to reduce stress.\n- Listening to calming music.\n- Applying peppermint or lavender essential oils, which can promote relaxation.\n- Ensuring adequate hydration by drinking water to prevent dehydration-related headaches.\n- Resting in a dark, quiet room to help alleviate headache symptoms.\n- Gentle massage of the temples and neck to relieve muscle tension.\n- Maintaining a regular sleep schedule to improve overall stress levels and headache frequency.\n\nRemember, it's important to combine these practices with a balanced lifestyle and consult with a healthcare provider if headaches or stress persist or worsen."

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [37]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [38]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [39]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nThese exercises are gentle and aimed at relieving discomfort and preventing future episodes. However, it's best to cons

In [40]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a profound impact on overall health. According to the provided information, sleep is crucial for physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep (7-9 hours for adults) helps support the immune system, maintain hormonal balance, and promote mental clarity. Conversely, poor or insufficient sleep can lead to issues such as fatigue, stress, impaired cognitive performance, and increased risk of health problems. Maintaining good sleep hygiene and creating an optimal sleep environment are essential steps to ensure quality sleep and, consequently, better overall health.'

In [41]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include practicing deep breathing, progressive muscle relaxation, grounding techniques (such as naming things you see, hear, feel, smell, and taste), taking short walks in nature, and listening to calming music. \n\nFor headaches, natural remedies include drinking plenty of water to stay hydrated, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, gentle massage of the temples and neck, using essential oils like peppermint or lavender, and maintaining a regular sleep schedule.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [42]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [43]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [44]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [45]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [46]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [47]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee towards your chest while keeping the other foot flat. Hold for 15-30 seconds each leg.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides.\n\nStarting gently with these movements can help alleviate discomfort and improve your lower back's strength and flexibility. If you're uncertain or expe

In [48]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly impacts overall health by supporting physical repair, mental well-being, and cognitive function. During sleep, the body heals tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep (typically 7-9 hours for adults) helps maintain a healthy immune system, improves mood, enhances focus, and reduces the risk of chronic conditions such as heart disease, diabetes, and obesity. Poor sleep can lead to fatigue, impaired immune function, mood disturbances, and increased vulnerability to illness. Therefore, prioritizing quality sleep is essential for overall health and wellness.'

In [49]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing relaxation techniques such as deep breathing exercises (inhaling for 4 counts, holding for 4, exhaling for 4), progressive muscle relaxation, and mindful grounding techniques (naming things you see, hear, feel, smell, and taste). Additionally, engaging in short walks, listening to calming music, and applying peppermint or lavender essential oils can help alleviate headaches. For hydration and relief from tension, drinking water, applying cold or warm compresses to the head or neck, and resting in a dark, quiet room are effective. Regular sleep routines, stress management practices, and hobbies can also support overall stress reduction and headache prevention.'

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

Semantic chunking (as in LangChain’s SemanticChunker) works by:
    1. Embedding each sentence
    2. Computing similarity between adjacent sentences
    3. Splitting when similarity drops below a threshold (percentile, standard deviation, etc.)

Problems with FAQs and Similar Content:
1. High similarity everywhere
Many FAQ items share structure (“What is…?”, “How do I…?”) and phrasing. Embeddings will be very similar across items, so the algorithm tends to see few “breaks” and may merge many unrelated Q&A pairs into large chunks.

2. Weak topic boundaries
The method relies on similarity drops between adjacent sentences. In FAQs, transitions between topics can be small, so boundaries are hard to detect.

3. Short chunks
Each Q&A is often 1–2 sentences. With a percentile-based threshold, the distribution of similarities can be compressed, so the chosen percentile may not separate topics well.

4. Over-merging
The result can be large chunks that mix unrelated questions, which hurts retrieval precision.

How to Adjust the Algorithm:
1. Use structure-aware chunking
Treat each Q&A pair as its own unit. Split on patterns like Q:, A:, or numbered questions, and only use semantic similarity within those units if needed.

2. Raise the breakpoint threshold
With breakpoint_threshold_type="percentile", try a higher percentile (e.g., 90 instead of 80) so more similarity drops count as breaks. That makes chunking more conservative and reduces over-merging.

3. Try other threshold types
standard_deviation or interquartile can behave differently on narrow similarity distributions. Experiment to see which gives more sensible boundaries.

4. Hybrid approach
First split by structure (e.g., each Q&A pair), then optionally use semantic chunking only within longer sections.

5. Minimum chunk size
Add a rule: don’t merge chunks below a certain size (e.g., 2–3 sentences). That keeps each FAQ item as a separate chunk even when similarities are high.

6. Different embeddings
Some models separate subtle topic differences better. Trying another embedding model can improve boundary detection.

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [50]:
# Step 1: Create Golden Dataset using Ragas Synthetic Data Generation
import time
import pandas as pd
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.metrics import LLMContextRecall, ContextEntityRecall
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Setup for synthetic data generation
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini", temperature=0))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
golden_dataset = generator.generate_with_langchain_docs(raw_docs, testset_size=10)

print(f"Generated {len(golden_dataset)} test samples")
golden_dataset.to_pandas().head(3)

/var/folders/11/zm4qrjwd3299fqklnymfw9vh0000gn/T/ipykernel_47297/3359780614.py:8: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, ContextEntityRecall
/var/folders/11/zm4qrjwd3299fqklnymfw9vh0000gn/T/ipykernel_47297/3359780614.py:8: DeprecationWarning: Importing ContextEntityRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextEntityRecall
  from ragas.metrics import LLMContextRecall, ContextEntityRecall
/var/folders/11/zm4qrjwd3299fqklnymfw9vh0000gn/T/ipykernel_47297/3359780614.py:12: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/6 [00:00<?, ?it/s]

...

Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

Generated 11 test samples


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,why strength training important for health and...,[The Personal Wellness Guide\nA Comprehensive ...,Strength training is one of the four main type...,Wellness Enthusiast,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
1,How should Neck Rolls be performed to relieve ...,[Chapter 2: Exercises for Common Problems\n\nL...,Neck Rolls should be performed by slowly rolli...,Wellness Enthusiast,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
2,How can magnesium supplements be used as a nat...,[Chapter 4: Fundamentals of Healthy Eating A b...,Magnesium supplements are mentioned as a natur...,Wellness Enthusiast,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer


In [57]:
# Step 2: Evaluate each retriever with Ragas retriever metrics (Context Recall, Context Entity Recall)
# We also track latency for each retriever. Cost can be viewed in LangSmith if LANGCHAIN_TRACING_V2 is set.

RETRIEVER_METRICS = [LLMContextRecall(), ContextEntityRecall()]
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
run_config = RunConfig(timeout=360)

retriever_chains = [
    ("Naive", naive_retrieval_chain),
    ("BM25", bm25_retrieval_chain),
    ("Contextual Compression (Rerank)", contextual_compression_retrieval_chain),
    ("Multi-Query", multi_query_retrieval_chain),
    ("Parent Document", parent_document_retrieval_chain),
    ("Ensemble", ensemble_retrieval_chain),
]

results_list = []
base_df = golden_dataset.to_pandas()

for retriever_name, chain in retriever_chains:
    print(f"\nEvaluating: {retriever_name}...")
    retrieved_contexts_list = []
    response_list = []
    total_latency = 0.0

    for _, row in base_df.iterrows():
        start = time.perf_counter()
        response = chain.invoke({"question": row["user_input"]})
        elapsed = time.perf_counter() - start
        total_latency += elapsed
        resp_content = response["response"].content if hasattr(response["response"], "content") else str(response["response"])
        retrieved_contexts_list.append([doc.page_content for doc in response["context"]])
        response_list.append(resp_content)

    avg_latency_ms = (total_latency / len(base_df)) * 1000

    eval_df = base_df.copy()
    eval_df["retrieved_contexts"] = retrieved_contexts_list
    eval_df["response"] = response_list
    # Fix NaN in string columns - Ragas SingleTurnSample expects strings, not float nan
    for col in ["persona_name", "query_style", "query_length"]:
        if col in eval_df.columns:
            eval_df[col] = eval_df[col].fillna("").astype(str)
    eval_dataset = EvaluationDataset.from_pandas(eval_df)

    ragas_result = evaluate(
        dataset=eval_dataset,
        metrics=RETRIEVER_METRICS,
        llm=evaluator_llm,
        run_config=run_config
    )

    # EvaluationResult uses bracket access and returns per-sample lists; compute mean
    scores_df = pd.DataFrame(ragas_result.scores)
    context_recall = float(scores_df["context_recall"].mean()) if "context_recall" in scores_df.columns else 0
    context_entity_recall = float(scores_df["context_entity_recall"].mean()) if "context_entity_recall" in scores_df.columns else 0

    results_list.append({
        "Retriever": retriever_name,
        "Context Recall": context_recall,
        "Context Entity Recall": context_entity_recall,
        "Avg Latency (ms)": round(avg_latency_ms, 2),
    })
    print(f"  Context Recall: {context_recall:.4f}, Entity Recall: {context_entity_recall:.4f}, Latency: {avg_latency_ms:.0f}ms")

/var/folders/11/zm4qrjwd3299fqklnymfw9vh0000gn/T/ipykernel_47297/1717781001.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))



Evaluating: Naive...


Evaluating:   0%|          | 0/22 [00:00<?, ?it/s]

  Context Recall: 1.0000, Entity Recall: 0.3474, Latency: 1966ms

Evaluating: BM25...


Evaluating:   0%|          | 0/22 [00:00<?, ?it/s]

...

  Context Recall: 0.9697, Entity Recall: 0.3493, Latency: 3440ms

Evaluating: Parent Document...


Evaluating:   0%|          | 0/22 [00:00<?, ?it/s]

  Context Recall: 0.9697, Entity Recall: 0.4067, Latency: 2410ms

Evaluating: Ensemble...


Evaluating:   0%|          | 0/22 [00:00<?, ?it/s]

  Context Recall: 1.0000, Entity Recall: 0.2885, Latency: 4266ms


In [58]:
# Step 3: Compile results into a table
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values("Context Recall", ascending=False).reset_index(drop=True)
print("Retriever Evaluation Results (sorted by Context Recall):")
results_df

Retriever Evaluation Results (sorted by Context Recall):


,Retriever,Context Recall,Context Entity Recall,Avg Latency (ms)
0,Naive,1.000000,0.347439,1966.30
1,Ensemble,1.000000,0.288502,4265.70
2,Multi-Query,0.969697,0.349325,3440.04
3,Parent Document,0.969697,0.406675,2409.72
4,Contextual Compression (Rerank),0.878788,0.380223,1865.75
5,BM25,0.662121,0.299893,1513.57


### Analysis: Which Retriever is Best for This Data?

#### Best Overall: Naive or Parent Document

**Naive** is the strongest default choice:
- **Context Recall:** 1.0 (highest)
- **Latency:** ~2 seconds (reasonable)
- **Cost:** Embeddings only (no LLM or Cohere API calls)

**Parent Document** is best when entity-level accuracy matters:
- **Context Entity Recall:** 0.41 (highest)
- **Context Recall:** 0.97
- **Latency:** ~2.4 seconds
- Useful when questions involve specific entities (nutrients, conditions, techniques)

#### Performance Summary

| Retriever | Context Recall | Entity Recall | Latency |
|-----------|----------------|---------------|---------|
| Naive | 1.00 | 0.35 | ~2.0s |
| Ensemble | 1.00 | 0.29 | ~4.3s |
| Multi-Query | 0.97 | 0.35 | ~3.4s |
| Parent Document | 0.97 | **0.41** | ~2.4s |
| Contextual Compression | 0.88 | 0.38 | ~1.9s |
| BM25 | 0.66 | 0.30 | ~1.5s |

#### When to Use Alternatives

- **BM25** — Lowest latency (~1.5s) and cost; no embeddings at query time. Use when speed and cost are the main constraints.
- **Ensemble** — Highest robustness (1.0 recall) but slowest (~4.3s) and most expensive.
- **Multi-Query** — Good for varied query phrasings; adds LLM cost for query expansion.
- **Contextual Compression** — Reranking improves precision; adds Cohere API cost.